# Генерация ответов Seq2Seq LSTM — версия 2.0

    **Цель:** Понять encoder/decoder, teacher forcing и перенос состояний LSTM.

    **Алгоритм:** Рекуррентная модель генерирует следующий токен. Это небольшой учебный Seq2Seq, не LLM и не трансформер.

    Учебная модернизация 2026-10-01. Код адаптирован с Python 3.7 на Python 3.12. Исходные алгоритмы сохранены; историческая
    версия доступна в Git на коммите `a842320`. Исправления учебных ошибок описаны
    в README. Это учебное пособие, не промышленная система и не benchmark.

    По умолчанию: CPU, короткий запуск, синтетические демонстрационные данные,
    без загрузок из сети. Реальные данные включаются отдельно: `MLLAB_DATA_SOURCE`.
    Синтетические результаты нельзя выдавать за качество на реальных данных.
    После изменения настроек перезапустите ядро и выполните ячейки сверху вниз.
    Окружение и структура данных описаны в `docs/ENVIRONMENTS.md` и `docs/DATA.md`.


In [ ]:
from pathlib import Path
import os, sys
start = Path(os.environ.get('MLLAB_ROOT', str(Path.cwd()))).resolve()
root = next((p for p in (start, *start.parents) if (p / 'ml_lab' / 'runtime.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Откройте папку репозитория или задайте MLLAB_ROOT.')
sys.path.insert(0, str(root))
from ml_lab.runtime import settings
cfg = settings('text-generation-seq2seq')


## Пары вопрос–ответ
Локальный pairs.json заменяет личный Google Drive и зависимость от YAML Loader.


In [ ]:
from ml_lab.runtime import tensorflow_setup
tf = tensorflow_setup(cfg)
import json
import numpy as np
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
if cfg.source == 'demo':
    pairs = [{'question': f'вопрос о числе {i}', 'answer': f'это число {i}'} for i in range(40)]
elif cfg.source == 'local':
    pairs = json.loads((cfg.data / 'pairs.json').read_text(encoding='utf-8'))
else:
    raise ValueError('Выберите demo или local')
if len(pairs) < 10 or any(not p.get('question') or not p.get('answer') for p in pairs):
    raise ValueError('Нужны минимум 10 непустых пар question/answer.')
train_idx, rest_idx = train_test_split(np.arange(len(pairs)), test_size=.4, random_state=cfg.seed)
val_idx, test_idx = train_test_split(rest_idx, test_size=.5, random_state=cfg.seed)
vectorizer = layers.TextVectorization(max_tokens=15000, output_mode='int', output_sequence_length=16 if cfg.quick else 64)
vocabulary_text = [value for i in train_idx for value in [pairs[i]['question'], 'startseq ' + pairs[i]['answer'] + ' endseq']]
vectorizer.adapt(tf.constant(vocabulary_text))
vocabulary = vectorizer.get_vocabulary()
def prepare(indices):
    questions = vectorizer(tf.constant([pairs[i]['question'] for i in indices])).numpy()
    answers = vectorizer(tf.constant(['startseq ' + pairs[i]['answer'] + ' endseq' for i in indices])).numpy()
    return questions, answers[:, :-1], answers[:, 1:]
x_train, d_train, y_train = prepare(train_idx)
x_val, d_val, y_val = prepare(val_idx)
x_test, d_test, y_test = prepare(test_idx)


## Encoder и decoder
Sparse loss не создаёт огромный one-hot массив; padding не участвует в обучении.


In [ ]:
units = 16 if cfg.quick else 200
encoder_input = layers.Input(shape=(None,), dtype='int32')
encoder_embedding = layers.Embedding(len(vocabulary), units, mask_zero=True)(encoder_input)
_, h, c = layers.LSTM(units, return_state=True)(encoder_embedding)
decoder_input = layers.Input(shape=(None,), dtype='int32')
embedding = layers.Embedding(len(vocabulary), units, mask_zero=True)
decoder_lstm = layers.LSTM(units, return_sequences=True, return_state=True)
decoded, _, _ = decoder_lstm(embedding(decoder_input), initial_state=[h, c])
dense = layers.Dense(len(vocabulary), activation='softmax')
model = Model([encoder_input, decoder_input], dense(decoded))
model.compile(optimizer=tf.keras.optimizers.RMSprop(), loss='sparse_categorical_crossentropy')
model.fit([x_train, d_train], y_train, sample_weight=(y_train != 0).astype('float32'),
    validation_data=([x_val, d_val], y_val, (y_val != 0).astype('float32')),
    epochs=1 if cfg.quick else 10, batch_size=8, verbose=0)
print('test loss:', model.evaluate([x_test, d_test], y_test, sample_weight=(y_test != 0).astype('float32'), verbose=0))


## Авторегрессионный вывод
Тестовый вопрос фиксирован для автоматической проверки; интерактивный input необязателен.


In [ ]:
encoder = Model(encoder_input, [h, c])
state_h, state_c = layers.Input(shape=(units,)), layers.Input(shape=(units,))
decoded, next_h, next_c = decoder_lstm(embedding(decoder_input), initial_state=[state_h, state_c])
decoder = Model([decoder_input, state_h, state_c], [dense(decoded), next_h, next_c])
def answer(question, maximum=20):
    states = encoder(vectorizer(tf.constant([question])), training=False)
    token = np.array([[vocabulary.index('startseq')]], dtype='int32')
    result = []
    for _ in range(maximum):
        probabilities, next_h, next_c = decoder([tf.convert_to_tensor(token), *states], training=False)
        index = int(tf.argmax(probabilities[0, 0]).numpy())
        if index in (0, vocabulary.index('endseq')): break
        result.append(vocabulary[index])
        token = np.array([[index]], dtype='int32'); states = [next_h, next_c]
    return ' '.join(result)
question = pairs[int(test_idx[0])]['question']
print('Вопрос:', question, '\nОтвет:', answer(question))


## Результат и самостоятельная работа

    Выполненные ячейки показывают формы данных, обучение и результат алгоритма.
    Успешный короткий запуск проверяет работоспособность, а не достижение заданной точности.

    1. Объясните сдвиг входа/выхода decoder.
2. Проследите состояния h и c.
3. Измените размер Embedding и LSTM.
4. Объясните ограничение длины генерации и маскирование padding.

    Перед выводом о качестве используйте отдельные train/validation/test,
    документируйте источник данных и сравнивайте эксперименты при одинаковых настройках.
